# Pruning

Structured pruning of Gemma-3 270M: MLP channels, attention heads, and whole blocks.

In [1]:
import torch
import torch_pruning as tp

from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
import dotenv
from pathlib import Path

from scripts.load_prompts import load_prompts_from_json

dotenv.load_dotenv()

True

In [2]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
VALIDATION_PROMPTS_PATH = (PROJECT_ROOT / "data" / "raw" / "intro_prompts.json")
assert VALIDATION_PROMPTS_PATH.exists(), VALIDATION_PROMPTS_PATH

## Load Gemma-3 270M

In [3]:
login(token=os.getenv("HF_TOKEN"), add_to_git_credential=False)
device = "cpu"

MODEL_NAME = "google/gemma-3-270m-it"
DTYPE = torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE,).to(device)

print("Model loaded on:", model.device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
W1024 00:21:14.916000 54173 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Model loaded on: cpu


In [4]:
def inference(model, prompt):
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7
        )

    return inputs, tokenizer.decode(outputs[0], skip_special_tokens=True)

In [5]:
def evaluate_model(model, tokenizer):
    model.eval()
    device = next(model.parameters()).device

    validation_data, _ = load_prompts_from_json(VALIDATION_PROMPTS_PATH)

    total_loss = 0.0
    total_tokens = 0
    for prompt in validation_data:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        ).to(device)

        with torch.no_grad():
            outputs = model(
                **inputs,
                labels=inputs["input_ids"],
                use_cache=False,
            )

        token_count = inputs["input_ids"].shape[1] - 1
        total_loss += outputs.loss.item() * token_count
        total_tokens += token_count

    return { "loss": total_loss / total_tokens }

In [6]:
def describe_model(model):
    print(type(model).__name__)
    print(model.model.layers)

In [7]:
tokenized_prompt, answer = inference(model, prompt="Answer in one word what is 1 + 1")
print(answer)

Answer in one word what is 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 +


## Model structure

In [8]:
describe_model(model)

Gemma3ForCausalLM
ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,

In [9]:
print(f"Parameters before pruning: {tp.utils.count_params(model)}")

Parameters before pruning: 268098176


## 1. Smallest MLP channels

Delete MLP channels with the smallest L2 norm. Each channel is removed jointly from gate, up, and down.

In [10]:
from copy import deepcopy
from torch import nn


def forward_fn(model, input_ids):
    return model(input_ids=input_ids, use_cache=False, return_dict=False)[0]


def prune_channels(model, ratio=0.25):
    model = deepcopy(model)

    mlp_roots = {
        proj
        for block in model.model.layers
        for proj in (block.mlp.gate_proj, block.mlp.up_proj, block.mlp.down_proj)
    }

    ignored_layers = [
        module
        for module in model.modules()
        if isinstance(module, nn.Linear) and module not in mlp_roots
    ]

    pruner = tp.pruner.MagnitudePruner(
        model=model,
        example_inputs=tokenized_prompt["input_ids"].to(model.device),
        forward_fn=forward_fn,
        importance=tp.importance.GroupMagnitudeImportance(p=2),
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
        root_module_types=[nn.Linear],
        round_to=8,
    )

    pruner.step()
    return model

In [11]:
pruned_model = prune_channels(model, ratio=0.1)

/Users/mac/PycharmProjects/small-LLM/.venv/lib/python3.14/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.14.input_layernorm.weight', 'model.layers.17.post_attention_layernorm.weight', 'model.layers.4.post_attention_layernorm.weight', 'model.layers.6.self_attn.q_norm.weight', 'model.layers.9.self_attn.q_norm.weight', 'model.layers.8.self_attn.k_norm.weight', 'model.layers.7.input_layernorm.weight', 'model.layers.15.post_feedforward_layernorm.weight', 'model.layers.16.post_attention_layernorm.weight', 'model.layers.17.post_feedforward_layernorm.weight', 'model.layers.10.self_attn.k_norm.weight', 'model.layers.1.post_feedforward_layernorm.weight', 'model.layers.15.input_layernorm.weight', 'model.layers.4.input_layernorm.weight', 'model.layers.14.pre_feedforward_layernorm.weight', 'model.layers.9.pre_feedforward_layernorm.weight', 'model.layers.3.post_feedforward_layernorm.weight', 'model.layers.9.input_layernorm.weight', 'mo

In [12]:
print(inference(pruned_model, "Hello, how are you?")[1])

Hello, how are you? ***ign

 AS
 on   biz,
 or
the- 2 and many b- result B
 and for that,
 and best
,
,  more open- KG-S-p-un, 
   
for b  
  ,
 &, &  the  or
   or
    ,  more  in- 2, &  
         at the  &so,



In [13]:
print(pruned_model.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=1840, bias=False)
      (up_proj): Linear(in_features=640, out_features=1840, bias=False)
      (down_proj): Linear(in_features=1840, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [14]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model)}")

Parameters after pruning: 260909696


### Loss

In [15]:
print("Original:", evaluate_model(model, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model, tokenizer))

Original: {'loss': 3.6954224108353904}
Pruned:   {'loss': 23.384014740197554}


**Result.** Smallest-norm channels are a poor choice. Loss increased about 6x.

## 2. Largest MLP channels

Same MLP cut, but delete the channels with the largest L2 norm.

In [16]:
def prune_model_2(model, ratio=0.1):
    model = deepcopy(model)

    mlp_roots = {
        proj
        for block in model.model.layers
        for proj in (block.mlp.gate_proj, block.mlp.up_proj, block.mlp.down_proj)
    }

    ignored_layers = [
        module
        for module in model.modules()
        if isinstance(module, nn.Linear) and module not in mlp_roots
    ]

    pruner = tp.MagnitudePruner(
        model=model,
        example_inputs=tokenized_prompt["input_ids"].to(model.device),
        forward_fn=forward_fn,
        importance=tp.importance.GroupMagnitudeImportance(p=2, normalizer=lambda s: -s),
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
        root_module_types=[nn.Linear],
        round_to=8,
    )

    pruner.step()
    return model

In [17]:
pruned_model2 = prune_model_2(model, ratio=0.1)

/Users/mac/PycharmProjects/small-LLM/.venv/lib/python3.14/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.2.input_layernorm.weight', 'model.layers.2.post_feedforward_layernorm.weight', 'model.layers.4.self_attn.k_norm.weight', 'model.layers.9.pre_feedforward_layernorm.weight', 'model.layers.8.input_layernorm.weight', 'model.layers.13.self_attn.q_norm.weight', 'model.layers.14.input_layernorm.weight', 'model.layers.14.pre_feedforward_layernorm.weight', 'model.layers.17.self_attn.k_norm.weight', 'model.layers.12.input_layernorm.weight', 'model.layers.2.post_attention_layernorm.weight', 'model.layers.4.post_feedforward_layernorm.weight', 'model.layers.15.self_attn.k_norm.weight', 'model.layers.13.post_feedforward_layernorm.weight', 'model.layers.3.input_layernorm.weight', 'model.layers.11.post_attention_layernorm.weight', 'model.layers.5.pre_feedforward_layernorm.weight', 'model.layers.0.self_attn.k_norm.weight', 'model.layer

In [18]:
print(inference(pruned_model2, "Hello, how are you?")[1])

Hello, how are you?归档品?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…? начални тела?…?…口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口?..口


In [19]:
print(pruned_model2.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=1840, bias=False)
      (up_proj): Linear(in_features=640, out_features=1840, bias=False)
      (down_proj): Linear(in_features=1840, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [20]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model2)}")

Parameters after pruning: 260909696


### Loss

In [21]:
print("Original:", evaluate_model(model, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model2, tokenizer))

Original: {'loss': 3.6954224108353904}
Pruned:   {'loss': 9.607682716328165}


**Result.** Better than the smallest-norm cut, but loss is still about 3x worse.

## 3. Random MLP channels

Delete random MLP channels, jointly from gate, up, and down.

In [22]:
def prune_model3(model, ratio=0.1, seed=42):
    model = deepcopy(model)

    torch.manual_seed(seed)

    for block in model.model.layers:
        n_prune = int(block.mlp.gate_proj.out_features * ratio)
        idx = torch.randperm(block.mlp.gate_proj.out_features)[:n_prune].tolist()

        tp.prune_linear_out_channels(block.mlp.gate_proj, idx)
        tp.prune_linear_out_channels(block.mlp.up_proj, idx)
        tp.prune_linear_in_channels(block.mlp.down_proj, idx)

    return model

In [23]:
pruned_model3 = prune_model3(model)

In [24]:
print(inference(pruned_model3, "Hello, how are you?")[1])

Hello, how are you?
Is there a chance of rain tomorrow evening?
I'm trying to find out what's going on with the news about the new ringgit (Silver Crystal Entacson).
I've just learned about it, but I'm not sure about the details.
What are your thoughts on what will happen if the ringgit falls?
I'm planning on to go out tonight, but I'm not sure if I'll see what happens.
I


In [25]:
print(pruned_model3.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=1844, bias=False)
      (up_proj): Linear(in_features=640, out_features=1844, bias=False)
      (down_proj): Linear(in_features=1844, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [26]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model3)}")

Parameters after pruning: 261047936


In [27]:
print("Original:", evaluate_model(model, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model3, tokenizer))

Original: {'loss': 3.6954224108353904}
Pruned:   {'loss': 6.65133057495822}


**Result.** Best MLP cut so far. Loss rose by almost 2x.

## 4. Attention heads

Remove whole query heads from `q_proj` and the matching inputs of `o_proj`.

In [28]:
def prune_heads(model, heads_num):
    model = deepcopy(model)

    head_size = model.config.head_dim

    delete_range = [i for head in heads_num for i in range(head * head_size, (head + 1) * head_size)]

    for block in model.model.layers:
        attn = block.self_attn
        q_proj, o_proj = attn.q_proj, attn.o_proj
        tp.prune_linear_out_channels(q_proj, delete_range)
        tp.prune_linear_in_channels(o_proj, delete_range)

        model.config.num_attention_heads -= len(heads_num)

    return model

In [29]:
pruned_model4 = prune_heads(model, [0])

In [30]:
print(pruned_model4.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=768, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=768, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )
)


In [31]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model4)}")

Parameters after pruning: 262199936


In [32]:
print("Original:", evaluate_model(model, tokenizer))

for head in range(model.config.num_attention_heads):
    new_model = prune_heads(model, [head])
    print(f"Pruned (deleted head: {head}):  ", evaluate_model(new_model, tokenizer))

Original: {'loss': 3.6954224108353904}
Pruned (deleted head: 0):   {'loss': 6.565630740963894}
Pruned (deleted head: 1):   {'loss': 10.337294565594714}
Pruned (deleted head: 2):   {'loss': 6.543373971918355}
Pruned (deleted head: 3):   {'loss': 5.224041224044302}


### Heads 0 and 2 together

In [33]:
pruned_model_head02 = prune_heads(model, [0, 2])
print("Pruned (deleted head: [0, 2]):  ", evaluate_model(pruned_model_head02, tokenizer))

Pruned (deleted head: [0, 2]):   {'loss': 11.600089723130932}


**Result.** Two heads hurt much more than one. Keep a single-head cut.

## 5. Whole blocks

The model has 18 decoder blocks. Delete a whole block.

In [34]:
def prune_blocks(model, blocks_num):
    model = deepcopy(model)
    drop = set(blocks_num)

    model.model.layers = nn.ModuleList(block for i, block in enumerate(model.model.layers) if i not in drop)
    model.config.layer_types = [kind for i, kind in enumerate(model.config.layer_types) if i not in drop]
    model.config.num_hidden_layers = len(model.model.layers)

    for new_index, block in enumerate(model.model.layers):
        block.layer_idx = new_index
        block.self_attn.layer_idx = new_index

    return model

In [35]:
pruned_model5 = prune_blocks(model, [0])

In [36]:
print(pruned_model5.model.layers)

ModuleList(
  (0-16): 17 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [37]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model5)}")

Parameters after pruning: 262524544


In [38]:
print("Original:", evaluate_model(model, tokenizer))

for block in range(len(model.model.layers)):
    new_model = prune_blocks(model, [block])
    print(f"Pruned (deleted block: {block}):  ", evaluate_model(new_model, tokenizer))

Original: {'loss': 3.6954224108353904}
Pruned (deleted block: 0):   {'loss': 13.46939960769985}
Pruned (deleted block: 1):   {'loss': 24.00942863277767}
Pruned (deleted block: 2):   {'loss': 5.346335839188617}
Pruned (deleted block: 3):   {'loss': 10.33031628365102}
Pruned (deleted block: 4):   {'loss': 4.252817714603051}
Pruned (deleted block: 5):   {'loss': 13.169324537982112}
Pruned (deleted block: 6):   {'loss': 4.299301384842914}
Pruned (deleted block: 7):   {'loss': 4.042909329740898}
Pruned (deleted block: 8):   {'loss': 4.242258813070214}
Pruned (deleted block: 9):   {'loss': 3.9987086551344913}
Pruned (deleted block: 10):   {'loss': 5.02773419579734}
Pruned (deleted block: 11):   {'loss': 5.926620708600335}
Pruned (deleted block: 12):   {'loss': 5.745852270851964}
Pruned (deleted block: 13):   {'loss': 6.105523883648541}
Pruned (deleted block: 14):   {'loss': 5.268271547037622}
Pruned (deleted block: 15):   {'loss': 5.201445143507875}
Pruned (deleted block: 16):   {'loss': 4.6

### Several blocks

In [39]:
pruned_model_blocks = prune_blocks(model, [7, 9])
print("Pruned (deleted block: [7, 9]):  ", evaluate_model(pruned_model_blocks, tokenizer))
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model_blocks)}")

Pruned (deleted block: [7, 9]):   {'loss': 4.5462598024503045}
Parameters after pruning: 256950912


**Result.** About 11M fewer parameters and loss raised only by 20%. Dropping blocks is worth keeping.